# ODI to Databricks Migration: WC_MERCURY_BADGE_DETAILS_D

**Source File:** WC_MERCURY_BADGE_DETAILS_D.txt
**Conversion Timestamp:** 2024-07-30T12:00:00Z

This notebook migrates the ODI session for `WC_MERCURY_BADGE_DETAILS_D` to Databricks Spark SQL. It handles staging, flow, and target table merges, including data type conversions, function mappings, and adherence to Databricks Delta Lake best practices.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "ETL Current Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_ROW_WID_PARAM", "0", "ETL Row WID")

## ETL Parameters

In [ ]:
-- MAGIC %sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type,
    '${DATASOURCE_NUM_ID}' AS datasource_num_id,
    '${ETL_PROC_WID}' AS etl_proc_wid,
    '${ODI_SESS_NO}' AS odi_sess_no,
    to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time_param,
    to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time_param,
    CAST('${ETL_ROW_WID_PARAM}' AS BIGINT) AS etl_row_wid_param;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2}: Get ETL last extract time from wc_etl_parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT
    COALESCE(
        MAX(T.last_extract_time),
        to_timestamp('1900-01-01', 'yyyy-MM-dd HH:mm:ss')
    ) AS etl_last_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters AS T
WHERE T.etl_job_type = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {3}: Get ETL current extract time from wc_etl_parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT
    COALESCE(
        MAX(T.current_extract_time),
        to_timestamp('9999-12-31', 'yyyy-MM-dd HH:mm:ss')
    ) AS etl_current_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters AS T
WHERE T.etl_job_type = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {4}:
Select etl_last_extract_time (duplicate view
from {2} - keeping for clarity)
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time_dup AS
SELECT etl_last_extract_time
FROM v_etl_last_extract_time;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {5}:
Select etl_current_extract_time (duplicate view
from {3} - keeping for clarity)
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time_dup AS
SELECT etl_current_extract_time
FROM v_etl_current_extract_time;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {6}: Select ROW_WID from wc_etl_parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT MAX(T.row_wid) AS etl_row_wid
FROM workspace.prxbi_dw.wc_etl_parameters AS T
WHERE T.etl_job_type = '${ETL_JOB_TYPE}';

In [ ]:
import pandas as pd
df_params = spark.sql("SELECT *
FROM v_etl_parameters").toPandas()
display(df_params)

## Staging Table (C$)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {30}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_wc_mercury_badge_ts_staging;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.prxbi_dw.c_wc_mercury_badge_ts_staging (
	ID	STRING,
	BADGELOCATION	STRING,
	BADGETOKEN	STRING,
	BADGEVERSION	BIGINT,
	CONTACTEMAIL	STRING,
	CONTACTFIRSTNAME	STRING,
	CONTACTJOBTITLE	STRING,
	CONTACTLASTNAME	STRING,
	CONTACTPERSONRXMASTERID	STRING,
	CREATEDBYREGISTRATIONTYPE	STRING,
	CREATEDBYTYPE	STRING,
	CULTURE	STRING,
	CUSTOMERTYPE	STRING,
	EVENTEDITIONGBSCODE	STRING,
	ISBADGEUPDATE	STRING,
	MARKETINGPREFERENCESPROMPTREQU	STRING,
	ORGANISATIONCITY	STRING,
	ORGANISATIONCOUNTRYCODE	STRING,
	ORGANISATIONDISPLAYNAME	STRING,
	ORGANISATIONRXMASTERID	STRING,
	ORGANISATIONSTATE	STRING,
	PARTICIPATINGORGANISATIONID	STRING,
	PRODUCTCODE	STRING,
	QRCODECONTENT	STRING,
	REGISTRATIONID	STRING,
	STATUS	BIGINT,
	SUPPORTSTAFFCOMPANYADDRESS	STRING,
	SUPPORTSTAFFCOMPANYNAME	STRING,
	SUPPORTSTAFFMOBILEPHONE	STRING,
	SUPPORTSTAFFREPORTSTO	STRING,
	SUPPORTSTAFFSTANDS	STRING,
	SUPPORTSTAFFUSERACCESS	STRING,
	VERSIONNUMBER	BIGINT,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging table with ROW_NUMBER() dedup (F.12 fix)
INSERT INTO workspace.prxbi_dw.c_wc_mercury_badge_ts_staging
SELECT
    ID,
    BADGELOCATION,
    BADGETOKEN,
    BADGEVERSION,
    CONTACTEMAIL,
    CONTACTFIRSTNAME,
    CONTACTJOBTITLE,
    CONTACTLASTNAME,
    CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE,
    CREATEDBYTYPE,
    CULTURE,
    CUSTOMERTYPE,
    EVENTEDITIONGBSCODE,
    ISBADGEUPDATE,
    MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
    ORGANISATIONCITY,
    ORGANISATIONCOUNTRYCODE,
    ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID,
    ORGANISATIONSTATE,
    PARTICIPATINGORGANISATIONID,
    PRODUCTCODE,
    QRCODECONTENT,
    REGISTRATIONID,
    STATUS,
    SUPPORTSTAFFCOMPANYADDRESS,
    SUPPORTSTAFFCOMPANYNAME,
    SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO,
    SUPPORTSTAFFSTANDS,
    SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ID
            ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC
        ) AS rn
    FROM workspace.prxbi_ts.wc_mercury_badge_ts
    WHERE
        INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
        AND INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
) AS deduped_source
WHERE rn = 1;

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS c_staging_row_count
FROM workspace.prxbi_dw.c_wc_mercury_badge_ts_staging;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: Gather table statistics (converted to OPTIMIZE)
OPTIMIZE workspace.prxbi_dw.c_wc_mercury_badge_ts_staging;

## Flow Table (I$)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {80}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow
(
	ROW_WID	BIGINT,
	BADGE_ID	STRING,
	BADGE_LOCATION	STRING,
	BADGE_TOKEN	STRING,
	BADGE_VERSION	BIGINT,
	CONTACT_EMAIL	STRING,
	CONTACT_FIRST_NAME	STRING,
	CONTACT_LAST_NAME	STRING,
	CONTACT_JOB_TITLE	STRING,
	CONTACT_PERSON_ID	STRING,
	CREATION_REG_TYPE	STRING,
	CREATION_TYPE	STRING,
	CULTURE	STRING,
	CUSTOMER_TYPE	STRING,
	EVENT_EDITION_CODE	STRING,
	BADGE_UPDATE_FLG	STRING,
	MARKETING_PREF_PROMPT	STRING,
	ORG_NAME	STRING,
	ORG_CITY	STRING,
	ORG_COUNTRY	STRING,
	ORG_ID	STRING,
	ORG_STATE	STRING,
	PARTICIPATING_ORG_ID	STRING,
	PRODUCT_CODE	STRING,
	QR_CODE	STRING,
	REGISTRATION_ID	STRING,
	STATUS	BIGINT,
	STAFF_COMPANY_NAME	STRING,
	STAFF_COMPANY_ADDR	STRING,
	STAFF_PHONE_NUM	STRING,
	STAFF_REPORTING	STRING,
	STAFF_STANDS	STRING,
	STAFF_USER_ACCESS	STRING,
	VERSION_NUM	BIGINT,
	INTEGRATION_ID	STRING,
	DATASOURCE_NUM_ID	STRING,
	W_INSERT_DT	TIMESTAMP,
	W_UPDATE_DT	TIMESTAMP,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	FIRSTSCANNEDDATE_FLG	STRING,
	LASTPRINTEDDATE_FLG	STRING,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING,
	PACKAGE_NAME	STRING,
	IND_UPDATE	STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}: Insert into flow table
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow
(
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE,
    W_INSERT_DT,
    W_UPDATE_DT
)
SELECT
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    S.IND_UPDATE,
    current_timestamp(), -- W_INSERT_DT
    current_timestamp()  -- W_UPDATE_DT
FROM (
    SELECT
        JOIN1_A.ID AS BADGE_ID,
        JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
        JOIN1_A.CULTURE AS CULTURE,
        JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT AS QR_CODE,
        JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
        JOIN1_A.STATUS AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
        JOIN1_A.ID AS INTEGRATION_ID,
        CAST(${DATASOURCE_NUM_ID} AS STRING) AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
        CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME,
        'I' AS IND_UPDATE
    FROM workspace.prxbi_dw.c_wc_mercury_badge_ts_staging AS JOIN1_A
    LEFT OUTER JOIN (
        SELECT
            WC_BADGE_PRODUCT_D_1.ID AS ID,
            WC_BADGE_PRODUCT_D_1.SKU AS SKU,
            WC_BADGE_PRODUCT_D_1.NAME AS NAME,
            WC_BADGE_PRODUCT_D_1.COL AS COL,
            WC_BADGE_PRODUCT_D_1.SKU_1 AS SKU_1,
            WC_BADGE_PRODUCT_D_1.NAME_1 AS NAME_1
        FROM (
            SELECT
                WC_BADGE_PRODUCT_D.ID AS ID,
                WC_BADGE_PRODUCT_D.SKU AS SKU,
                WC_BADGE_PRODUCT_D.NAME AS NAME,
                rank() OVER (PARTITION BY WC_BADGE_PRODUCT_D.SKU ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
            FROM workspace.prxbi_dw.wc_badge_product_d AS WC_BADGE_PRODUCT_D
        ) AS WC_BADGE_PRODUCT_D_1
        WHERE WC_BADGE_PRODUCT_D_1.COL = 1
    ) AS WC_BADGE_PRODUCT_D_2
    ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
) AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.prxbi_dw.wc_badge_details_d AS T
    WHERE
        T.INTEGRATION_ID = S.INTEGRATION_ID
        AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
        AND ((T.BADGE_ID = S.BADGE_ID) OR (T.BADGE_ID IS NULL AND S.BADGE_ID IS NULL))
        AND ((T.BADGE_LOCATION = S.BADGE_LOCATION) OR (T.BADGE_LOCATION IS NULL AND S.BADGE_LOCATION IS NULL))
        AND ((T.BADGE_TOKEN = S.BADGE_TOKEN) OR (T.BADGE_TOKEN IS NULL AND S.BADGE_TOKEN IS NULL))
        AND ((T.BADGE_VERSION = S.BADGE_VERSION) OR (T.BADGE_VERSION IS NULL AND S.BADGE_VERSION IS NULL))
        AND ((T.CONTACT_EMAIL = S.CONTACT_EMAIL) OR (T.CONTACT_EMAIL IS NULL AND S.CONTACT_EMAIL IS NULL))
        AND ((T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME) OR (T.CONTACT_FIRST_NAME IS NULL AND S.CONTACT_FIRST_NAME IS NULL))
        AND ((T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME) OR (T.CONTACT_LAST_NAME IS NULL AND S.CONTACT_LAST_NAME IS NULL))
        AND ((T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE) OR (T.CONTACT_JOB_TITLE IS NULL AND S.CONTACT_JOB_TITLE IS NULL))
        AND ((T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID) OR (T.CONTACT_PERSON_ID IS NULL AND S.CONTACT_PERSON_ID IS NULL))
        AND ((T.CREATION_REG_TYPE = S.CREATION_REG_TYPE) OR (T.CREATION_REG_TYPE IS NULL AND S.CREATION_REG_TYPE IS NULL))
        AND ((T.CREATION_TYPE = S.CREATION_TYPE) OR (T.CREATION_TYPE IS NULL AND S.CREATION_TYPE IS NULL))
        AND ((T.CULTURE = S.CULTURE) OR (T.CULTURE IS NULL AND S.CULTURE IS NULL))
        AND ((T.CUSTOMER_TYPE = S.CUSTOMER_TYPE) OR (T.CUSTOMER_TYPE IS NULL AND S.CUSTOMER_TYPE IS NULL))
        AND ((T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE) OR (T.EVENT_EDITION_CODE IS NULL AND S.EVENT_EDITION_CODE IS NULL))
        AND ((T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG) OR (T.BADGE_UPDATE_FLG IS NULL AND S.BADGE_UPDATE_FLG IS NULL))
        AND ((T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT) OR (T.MARKETING_PREF_PROMPT IS NULL AND S.MARKETING_PREF_PROMPT IS NULL))
        AND ((T.ORG_NAME = S.ORG_NAME) OR (T.ORG_NAME IS NULL AND S.ORG_NAME IS NULL))
        AND ((T.ORG_CITY = S.ORG_CITY) OR (T.ORG_CITY IS NULL AND S.ORG_CITY IS NULL))
        AND ((T.ORG_COUNTRY = S.ORG_COUNTRY) OR (T.ORG_COUNTRY IS NULL AND S.ORG_COUNTRY IS NULL))
        AND ((T.ORG_ID = S.ORG_ID) OR (T.ORG_ID IS NULL AND S.ORG_ID IS NULL))
        AND ((T.ORG_STATE = S.ORG_STATE) OR (T.ORG_STATE IS NULL AND S.ORG_STATE IS NULL))
        AND ((T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID) OR (T.PARTICIPATING_ORG_ID IS NULL AND S.PARTICIPATING_ORG_ID IS NULL))
        AND ((T.PRODUCT_CODE = S.PRODUCT_CODE) OR (T.PRODUCT_CODE IS NULL AND S.PRODUCT_CODE IS NULL))
        AND ((T.QR_CODE = S.QR_CODE) OR (T.QR_CODE IS NULL AND S.QR_CODE IS NULL))
        AND ((T.REGISTRATION_ID = S.REGISTRATION_ID) OR (T.REGISTRATION_ID IS NULL AND S.REGISTRATION_ID IS NULL))
        AND ((T.STATUS = S.STATUS) OR (T.STATUS IS NULL AND S.STATUS IS NULL))
        AND ((T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME) OR (T.STAFF_COMPANY_NAME IS NULL AND S.STAFF_COMPANY_NAME IS NULL))
        AND ((T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR) OR (T.STAFF_COMPANY_ADDR IS NULL AND S.STAFF_COMPANY_ADDR IS NULL))
        AND ((T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM) OR (T.STAFF_PHONE_NUM IS NULL AND S.STAFF_PHONE_NUM IS NULL))
        AND ((T.STAFF_REPORTING = S.STAFF_REPORTING) OR (T.STAFF_REPORTING IS NULL AND S.STAFF_REPORTING IS NULL))
        AND ((T.STAFF_STANDS = S.STAFF_STANDS) OR (T.STAFF_STANDS IS NULL AND S.STAFF_STANDS IS NULL))
        AND ((T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS) OR (T.STAFF_USER_ACCESS IS NULL AND S.STAFF_USER_ACCESS IS NULL))
        AND ((T.VERSION_NUM = S.VERSION_NUM) OR (T.VERSION_NUM IS NULL AND S.VERSION_NUM IS NULL))
        AND ((T.MOBILEPHONE = S.MOBILEPHONE) OR (T.MOBILEPHONE IS NULL AND S.MOBILEPHONE IS NULL))
        AND ((T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE) OR (T.FIRSTSCANNEDDATE IS NULL AND S.FIRSTSCANNEDDATE IS NULL))
        AND ((T.LASTPRINTEDDATE = S.LASTPRINTEDDATE) OR (T.LASTPRINTEDDATE IS NULL AND S.LASTPRINTEDDATE IS NULL))
        AND ((T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG) OR (T.FIRSTSCANNEDDATE_FLG IS NULL AND S.FIRSTSCANNEDDATE_FLG IS NULL))
        AND ((T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG) OR (T.LASTPRINTEDDATE_FLG IS NULL AND S.LASTPRINTEDDATE_FLG IS NULL))
        AND ((T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE) OR (T.ACCESSVALIDITYMODIFIEDDATE IS NULL AND S.ACCESSVALIDITYMODIFIEDDATE IS NULL))
        AND ((T.CREATEDDATE = S.CREATEDDATE) OR (T.CREATEDDATE IS NULL AND S.CREATEDDATE IS NULL))
        AND ((T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE) OR (T.COMPANYPRODUCTCODE IS NULL AND S.COMPANYPRODUCTCODE IS NULL))
        AND ((T.PAYMENTSTATUS = S.PAYMENTSTATUS) OR (T.PAYMENTSTATUS IS NULL AND S.PAYMENTSTATUS IS NULL))
        AND ((T.PHOTOKEY = S.PHOTOKEY) OR (T.PHOTOKEY IS NULL AND S.PHOTOKEY IS NULL))
        AND ((T.PHOTOSOURCE = S.PHOTOSOURCE) OR (T.PHOTOSOURCE IS NULL AND S.PHOTOSOURCE IS NULL))
        AND ((T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE) OR (T.PHOTOSOURCETYPE IS NULL AND S.PHOTOSOURCETYPE IS NULL))
        AND ((T.PACKAGE_NAME = S.PACKAGE_NAME) OR (T.PACKAGE_NAME IS NULL AND S.PACKAGE_NAME IS NULL))
);

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS i_flow_row_count
FROM workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Create index (converted to
OPTIMIZE ZORDER BY)
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {120}: Gather table statistics (converted to OPTIMIZE)
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow;

## Mark Records for Update

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Update IND_UPDATE flag for existing records (F.10 multi-column IN predicate fix)
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT
        INTEGRATION_ID,
        DATASOURCE_NUM_ID
    FROM workspace.prxbi_dw.wc_badge_details_d
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target (WC_BADGE_DETAILS_D)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150} + {160} combined: Update existing records and insert new records (F.3 tuple-SET UPDATE and F.4 IDENTITY column fix)
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.BADGE_ID                      = S.BADGE_ID,
    T.BADGE_LOCATION                = S.BADGE_LOCATION,
    T.BADGE_TOKEN                   = S.BADGE_TOKEN,
    T.BADGE_VERSION                 = S.BADGE_VERSION,
    T.CONTACT_EMAIL                 = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME            = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME             = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE             = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID             = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE             = S.CREATION_REG_TYPE,
    T.CREATION_TYPE                 = S.CREATION_TYPE,
    T.CULTURE                       = S.CULTURE,
    T.CUSTOMER_TYPE                 = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE            = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG              = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT         = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME                      = S.ORG_NAME,
    T.ORG_CITY                      = S.ORG_CITY,
    T.ORG_COUNTRY                   = S.ORG_COUNTRY,
    T.ORG_ID                        = S.ORG_ID,
    T.ORG_STATE                     = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID          = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE                  = S.PRODUCT_CODE,
    T.QR_CODE                       = S.QR_CODE,
    T.REGISTRATION_ID               = S.REGISTRATION_ID,
    T.STATUS                        = S.STATUS,
    T.STAFF_COMPANY_NAME            = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR            = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM               = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING               = S.STAFF_REPORTING,
    T.STAFF_STANDS                  = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS             = S.STAFF_USER_ACCESS,
    T.VERSION_NUM                   = S.VERSION_NUM,
    T.MOBILEPHONE                   = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE              = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE               = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG          = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG           = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE    = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE                   = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE            = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS                 = S.PAYMENTSTATUS,
    T.PHOTOKEY                      = S.PHOTOKEY,
    T.PHOTOSOURCE                   = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE               = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME                  = S.PACKAGE_NAME,
    T.W_UPDATE_DT                   = current_timestamp()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    W_INSERT_DT,
    W_UPDATE_DT
) VALUES (
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    current_timestamp(), -- W_INSERT_DT
    current_timestamp()  -- W_UPDATE_DT
);

## Cleanup

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {180}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {210}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_wc_mercury_badge_ts_staging;

## Validation

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS final_target_row_count
FROM workspace.prxbi_dw.wc_badge_details_d;

In [ ]:
-- MAGIC %sql
SELECT * FROM workspace.prxbi_dw.wc_badge_details_d
WHERE W_UPDATE_DT >= current_date() - INTERVAL '1' DAY
ORDER BY W_UPDATE_DT DESC
LIMIT 100;

## Conversion Notes & Manual Actions Required

1.  **ROW_WID Handling**: The `ROW_WID` column in the target table `workspace.prxbi_dw.wc_badge_details_d` was originally populated by `WC_BADGE_DETAILS_D_SEQ.NEXTVAL`. In Databricks, it is recommended to define this column as `BIGINT GENERATED ALWAYS AS IDENTITY`. If this DDL change has been applied to the target table, `ROW_WID` has been explicitly removed from the `INSERT` clause in the final MERGE statement (SCEN_TASK_NO {150}/{160} combined), as `GENERATED ALWAYS AS IDENTITY` columns are auto-populated and cannot be explicitly inserted or updated. If the target table is not configured with `GENERATED ALWAYS AS IDENTITY`, this might need adjustment (e.g., generating a `uuid_bigint()` or `monotonically_increasing_id()` in the `SELECT` list and inserting it).
2.  **ETL Parameters**: The ODI `#GLOBAL.V_ETL_LAST_EXTRACT_TIME`, `#GLOBAL.V_ETL_CURRENT_EXTRACT_TIME`, `#GLOBAL.v_ETL_JOB_TYPE`, etc., have been converted to Databricks widgets (`${ETL_LAST_EXTRACT_TIME}`, `${ETL_JOB_TYPE}`). Ensure these widgets are set correctly before execution, especially `ETL_LAST_EXTRACT_TIME` and `ETL_CURRENT_EXTRACT_TIME` for incremental loads. Views `v_etl_last_extract_time` and `v_etl_current_extract_time` are created based on the `wc_etl_parameters` table and widget values.
3.  **Schema and Table Naming**: All Oracle schema names (`PRXBI_DW_SEP`, `PRXBI_TS_SEP`) have been converted to lowercase Databricks format (`workspace.prxbi_dw`, `workspace.prxbi_ts`). Staging and flow tables have been given meaningful names (e.g., `c_wc_mercury_badge_ts_staging`, `i_wc_badge_details_d_flow`).
4.  **ODI MAX Self-Join Dedup (F.12)**: The ODI pattern of using a self-join with independently aggregated `MAX` columns for deduplication in SCEN_TASK_NO {50} has been rewritten to use `ROW_NUMBER() OVER (PARTITION BY ID ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC) WHERE rn = 1`. This ensures deterministic single-row-per-key output in Spark.
5.  **Multi-Column IN Predicate (F.10)**: The `UPDATE ... WHERE (col1, col2) IN (SELECT ...)` statement in SCEN_TASK_NO {130} has been rewritten as a `MERGE` statement, which is the supported pattern for Delta Lake tables.
6.  **Oracle Tuple-SET Update (F.3)**: The Oracle specific `UPDATE T SET (col1, col2) = (SELECT ...)` syntax in SCEN_TASK_NO {150} has been rewritten as a `MERGE INTO ... WHEN MATCHED THEN UPDATE SET` statement.
7.  **Index and Statistics (SCEN_TASK_NO {60}, {110}, {120})**: Oracle `CREATE INDEX` and `DBMS_STATS.GATHER_TABLE_STATS` statements have been converted to Databricks `OPTIMIZE` commands. `OPTIMIZE ... ZORDER BY` is used for indexing, and standard `OPTIMIZE` is used for statistics collection. The `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` is included for `ZORDER BY` commands to prevent errors related to missing statistics (F.11).
8.  **Generic Oracle Functions**: `NVL2` has been replaced with `CASE WHEN ... IS NOT NULL THEN ... ELSE ... END`. `TO_TIMESTAMP` format strings have been updated to Spark SQL compatible formats (e.g., `YYYY-MM-DD HH24:MI:SS.FF` to `yyyy-MM-dd HH:mm:ss.SSSSSS`). `SYSTIMESTAMP` has been replaced with `current_timestamp()`.
9.  **Table `wc_etl_parameters`**: This table is assumed to exist in `workspace.prxbi_dw` and contain columns like `etl_job_type`, `last_extract_time`, `current_extract_time`, and `row_wid` for ETL parameter management. Manual creation or verification of this table is required if it doesn't exist.